# Step 3 — Tensor Preparation + Encoders + Survival Head

This notebook implements the first working multimodal architecture for:

**Bioprosthetic valve durability prediction**

It expects the outputs from Step 2b:

- `synthetic_multimodal_patient_year_v2_MODEL_READY.csv`
- `encoder_input_spec_v2.json`

## Architecture V1

```text
Static valve / patient features
        ↓
Valve Encoder (MLP)
        ↓
z_valve

Longitudinal physiology + valve hemodynamics
        ↓
Physiology Encoder (GRU)
        ↓
z_phys

Longitudinal medications
        ↓
Medication Encoder (GRU)
        ↓
z_med

[z_phys, z_med]
        ↓
Temporal Fusion
        ↓
z_patient

[z_patient, z_valve]
        ↓
Discrete-time Survival Head
        ↓
hazard(t) → S(t)
```

### Notes in V1

Raw clinical text is **not** used yet.

The structured note-derived signals (`note_dyspnea`, `note_valve_dysfunction`) are optional temporal context and can be appended to the physiology branch.

### Important leakage rule

All preprocessing statistics are fit **only on the training patients**.


In [1]:
from pathlib import Path
import json
import math
import copy

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

DATA_CSV = Path("synthetic_generator_outputs_v2/synthetic_multimodal_patient_year_v2_MODEL_READY.csv")
SPEC_JSON = Path("synthetic_generator_outputs_v2/encoder_input_spec_v2.json")

# If your files are next to the notebook instead, use:
# DATA_CSV = Path("synthetic_multimodal_patient_year_v2_MODEL_READY.csv")
# SPEC_JSON = Path("encoder_input_spec_v2.json")

print("Data:", DATA_CSV.resolve())
print("Spec:", SPEC_JSON.resolve())


ModuleNotFoundError: No module named 'torch'

## 1. Load dataset and encoder specification

In [ ]:
if not DATA_CSV.exists():
    raise FileNotFoundError(
        f"Model-ready CSV not found: {DATA_CSV.resolve()}\n"
        "Update DATA_CSV in the first code cell."
    )

if not SPEC_JSON.exists():
    raise FileNotFoundError(
        f"Encoder spec JSON not found: {SPEC_JSON.resolve()}\n"
        "Update SPEC_JSON in the first code cell."
    )

df = pd.read_csv(DATA_CSV)

with open(SPEC_JSON, "r", encoding="utf-8") as f:
    spec = json.load(f)

PATIENT_COL = spec["patient_id_column"]
TIME_COL = spec["time_column"]

STATIC_FEATURES = spec["static_features"]
PHYS_FEATURES = spec["physiology_features"]
MED_FEATURES = spec["medication_features"]
NOTE_FEATURES = spec.get("note_derived_features_v1_optional", [])
TARGETS = spec["targets"]

# valve_position is constant in the current synthetic cohort, so we remove it
# from the trainable static input while keeping it in the dataset as metadata.
STATIC_FEATURES_MODEL = [
    c for c in STATIC_FEATURES
    if c != "valve_position"
]

print("Shape:", df.shape)
print("Patients:", df[PATIENT_COL].nunique())
print("Static features:", len(STATIC_FEATURES_MODEL))
print("Physiology features:", len(PHYS_FEATURES))
print("Medication features:", len(MED_FEATURES))
print("Optional note-derived features:", NOTE_FEATURES)

display(df.head())


## 2. Patient-level train / validation / test split

The split is performed by **patient ID**, never by rows.

This prevents visits from the same patient appearing in both train and test.


In [ ]:
patient_table = (
    df[[PATIENT_COL, "event", "duration_months"]]
    .drop_duplicates(subset=[PATIENT_COL])
    .reset_index(drop=True)
)

# 70% train, 15% validation, 15% test.
gss1 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=SEED,
)

train_idx, temp_idx = next(
    gss1.split(
        patient_table,
        groups=patient_table[PATIENT_COL],
    )
)

train_patients = patient_table.iloc[train_idx][PATIENT_COL].tolist()
temp = patient_table.iloc[temp_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.50,
    random_state=SEED + 1,
)

val_idx, test_idx = next(
    gss2.split(
        temp,
        groups=temp[PATIENT_COL],
    )
)

val_patients = temp.iloc[val_idx][PATIENT_COL].tolist()
test_patients = temp.iloc[test_idx][PATIENT_COL].tolist()

print("Train patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Test patients:", len(test_patients))

for name, ids in [
    ("train", train_patients),
    ("val", val_patients),
    ("test", test_patients),
]:
    sub = patient_table[patient_table[PATIENT_COL].isin(ids)]
    print(
        name,
        "event rate =",
        round(float(sub["event"].mean()), 3),
        "| median duration =",
        round(float(sub["duration_months"].median()), 1),
    )


## 3. Fit preprocessing on TRAIN only

### Static branch
- numeric static variables → standardization
- categorical static variables → one-hot encoding

### Physiology branch
- numeric values → standardization using observed training values only
- missing values → zero after standardization
- a missingness mask is concatenated to the values

### Medication branch
- binary values kept as 0/1
- missing values → zero
- a medication observation mask is concatenated

### Time
A normalized `time_since_implant_months` channel is appended to both temporal branches.


In [ ]:
train_df = df[df[PATIENT_COL].isin(train_patients)].copy()

STATIC_NUMERIC = [
    c for c in STATIC_FEATURES_MODEL
    if pd.api.types.is_numeric_dtype(train_df[c])
]

STATIC_CATEGORICAL = [
    c for c in STATIC_FEATURES_MODEL
    if c not in STATIC_NUMERIC
]

print("Static numeric:", STATIC_NUMERIC)
print("Static categorical:", STATIC_CATEGORICAL)

# ---- static scaler
static_scaler = StandardScaler()

static_train_rows = (
    train_df
    .sort_values([PATIENT_COL, TIME_COL])
    .groupby(PATIENT_COL)
    .first()
    .reset_index()
)

static_scaler.fit(
    static_train_rows[STATIC_NUMERIC].astype(float)
)

# ---- static categorical encoder
try:
    static_ohe = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )
except TypeError:
    # for older scikit-learn
    static_ohe = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False,
    )

static_ohe.fit(
    static_train_rows[STATIC_CATEGORICAL].astype(str)
)

# ---- physiology per-feature train statistics
phys_train = train_df[PHYS_FEATURES].apply(
    pd.to_numeric,
    errors="coerce",
)

phys_mean = phys_train.mean()
phys_std = phys_train.std().replace(0, 1.0)

# Avoid NaN std if an entire feature were missing.
phys_mean = phys_mean.fillna(0.0)
phys_std = phys_std.fillna(1.0)

# ---- time scaling
TIME_SCALE = max(
    float(train_df[TIME_COL].max()),
    1.0,
)

print("Physiology channels:", len(PHYS_FEATURES))
print("Time scale:", TIME_SCALE)


## 4. Build one sample per patient

Each patient sample contains:

- `x_static`
- `x_phys [T, F_phys*2 + 1]`
- `x_med [T, F_med*2 + 1]`
- `sequence_mask [T]`
- `duration`
- `event`

The doubled temporal dimensions come from:

`values + missingness mask`

and `+1` is the normalized time channel.


In [ ]:
def transform_static(patient_df):
    first = (
        patient_df
        .sort_values(TIME_COL)
        .iloc[0]
    )

    num = first[STATIC_NUMERIC].astype(float).to_numpy().reshape(1, -1)
    num = static_scaler.transform(num)

    cat = (
        first[STATIC_CATEGORICAL]
        .astype(str)
        .to_numpy()
        .reshape(1, -1)
    )
    cat = static_ohe.transform(cat)

    return np.concatenate([num, cat], axis=1).astype(np.float32).squeeze(0)


def transform_phys(patient_df, include_notes=True):
    pdf = patient_df.sort_values(TIME_COL)

    values = (
        pdf[PHYS_FEATURES]
        .apply(pd.to_numeric, errors="coerce")
    )

    mask = values.notna().astype(np.float32)

    standardized = (
        (values - phys_mean) / phys_std
    ).fillna(0.0).astype(np.float32)

    parts = [
        standardized.to_numpy(),
        mask.to_numpy(),
    ]

    # Optional structured note-derived temporal context.
    if include_notes and NOTE_FEATURES:
        note_values = (
            pdf[NOTE_FEATURES]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
            .astype(np.float32)
            .to_numpy()
        )
        parts.append(note_values)

    time_channel = (
        pdf[TIME_COL]
        .astype(float)
        .to_numpy()
        .reshape(-1, 1)
        / TIME_SCALE
    ).astype(np.float32)

    parts.append(time_channel)

    return np.concatenate(parts, axis=1).astype(np.float32)


def transform_med(patient_df):
    pdf = patient_df.sort_values(TIME_COL)

    med_values = (
        pdf[MED_FEATURES]
        .apply(pd.to_numeric, errors="coerce")
    )

    med_mask = med_values.notna().astype(np.float32)

    med_filled = (
        med_values
        .fillna(0.0)
        .astype(np.float32)
    )

    time_channel = (
        pdf[TIME_COL]
        .astype(float)
        .to_numpy()
        .reshape(-1, 1)
        / TIME_SCALE
    ).astype(np.float32)

    return np.concatenate(
        [
            med_filled.to_numpy(),
            med_mask.to_numpy(),
            time_channel,
        ],
        axis=1,
    ).astype(np.float32)


def make_patient_sample(patient_df):
    pdf = patient_df.sort_values(TIME_COL)

    return {
        "patient_id": str(pdf[PATIENT_COL].iloc[0]),
        "x_static": transform_static(pdf),
        "x_phys": transform_phys(pdf, include_notes=True),
        "x_med": transform_med(pdf),
        "duration": float(pdf["duration_months"].iloc[0]),
        "event": float(pdf["event"].iloc[0]),
    }


sample = make_patient_sample(
    df[df[PATIENT_COL] == df[PATIENT_COL].iloc[0]]
)

print("Static shape:", sample["x_static"].shape)
print("Physiology sequence shape:", sample["x_phys"].shape)
print("Medication sequence shape:", sample["x_med"].shape)
print("Duration:", sample["duration"])
print("Event:", sample["event"])


## 5. PyTorch Dataset and padded batches

In [ ]:
class PatientSequenceDataset(Dataset):
    def __init__(self, dataframe, patient_ids):
        self.df = dataframe[
            dataframe[PATIENT_COL].isin(patient_ids)
        ].copy()

        self.patient_ids = sorted(
            self.df[PATIENT_COL]
            .astype(str)
            .unique()
            .tolist()
        )

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        pid = self.patient_ids[idx]

        pdf = self.df[
            self.df[PATIENT_COL].astype(str) == pid
        ]

        return make_patient_sample(pdf)


def collate_patient_batch(batch):
    batch_size = len(batch)
    max_t = max(x["x_phys"].shape[0] for x in batch)

    static_dim = batch[0]["x_static"].shape[0]
    phys_dim = batch[0]["x_phys"].shape[1]
    med_dim = batch[0]["x_med"].shape[1]

    x_static = np.zeros(
        (batch_size, static_dim),
        dtype=np.float32,
    )

    x_phys = np.zeros(
        (batch_size, max_t, phys_dim),
        dtype=np.float32,
    )

    x_med = np.zeros(
        (batch_size, max_t, med_dim),
        dtype=np.float32,
    )

    seq_mask = np.zeros(
        (batch_size, max_t),
        dtype=np.float32,
    )

    lengths = np.zeros(
        batch_size,
        dtype=np.int64,
    )

    duration = np.zeros(
        batch_size,
        dtype=np.float32,
    )

    event = np.zeros(
        batch_size,
        dtype=np.float32,
    )

    patient_ids = []

    for i, item in enumerate(batch):
        t = item["x_phys"].shape[0]

        x_static[i] = item["x_static"]
        x_phys[i, :t] = item["x_phys"]
        x_med[i, :t] = item["x_med"]
        seq_mask[i, :t] = 1.0

        lengths[i] = t
        duration[i] = item["duration"]
        event[i] = item["event"]
        patient_ids.append(item["patient_id"])

    return {
        "patient_id": patient_ids,
        "x_static": torch.tensor(x_static),
        "x_phys": torch.tensor(x_phys),
        "x_med": torch.tensor(x_med),
        "seq_mask": torch.tensor(seq_mask),
        "lengths": torch.tensor(lengths),
        "duration": torch.tensor(duration),
        "event": torch.tensor(event),
    }


train_ds = PatientSequenceDataset(df, train_patients)
val_ds = PatientSequenceDataset(df, val_patients)
test_ds = PatientSequenceDataset(df, test_patients)

train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_patient_batch,
)

val_loader = DataLoader(
    val_ds,
    batch_size=128,
    shuffle=False,
    collate_fn=collate_patient_batch,
)

test_loader = DataLoader(
    test_ds,
    batch_size=128,
    shuffle=False,
    collate_fn=collate_patient_batch,
)

batch = next(iter(train_loader))

print("x_static:", batch["x_static"].shape)
print("x_phys:", batch["x_phys"].shape)
print("x_med:", batch["x_med"].shape)
print("seq_mask:", batch["seq_mask"].shape)
print("lengths:", batch["lengths"][:10])


## 6. Define the three encoders

### Valve Encoder
Simple MLP over static valve/patient features.

### Physiology Encoder
GRU over longitudinal physiology + hemodynamics + observation masks + optional note-derived structured features.

### Medication Encoder
GRU over medication indicators + medication observation masks.

The last valid GRU state is used as the temporal embedding.


In [ ]:
class StaticValveEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, output_dim=32):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(hidden_dim, output_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class GRUSequenceEncoder(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim=64,
        output_dim=32,
        num_layers=1,
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )

        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, output_dim),
            nn.ReLU(),
        )

    def forward(self, x, lengths):
        # Pack sequences so padding does not affect the GRU.
        packed = nn.utils.rnn.pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        _, hidden = self.gru(packed)

        last_hidden = hidden[-1]

        return self.projection(last_hidden)


## 7. Fusion + discrete-time survival head

We use yearly survival bins up to 10 years.

The model predicts a hazard probability for each interval.

If:

`h_k = P(event in interval k | survived before interval k)`

then:

`S_k = Π(1 - h_j)` for `j <= k`.


In [ ]:
SURVIVAL_BIN_EDGES = np.arange(
    0,
    121,
    12,
    dtype=np.float32,
)

N_SURVIVAL_BINS = len(SURVIVAL_BIN_EDGES) - 1

print("Survival bins:", N_SURVIVAL_BINS)
print("Edges:", SURVIVAL_BIN_EDGES)


class MultimodalValveSurvivalModel(nn.Module):
    def __init__(
        self,
        static_input_dim,
        phys_input_dim,
        med_input_dim,
        valve_dim=32,
        phys_dim=32,
        med_dim=32,
        fusion_dim=64,
        n_survival_bins=N_SURVIVAL_BINS,
    ):
        super().__init__()

        self.valve_encoder = StaticValveEncoder(
            input_dim=static_input_dim,
            hidden_dim=64,
            output_dim=valve_dim,
        )

        self.physiology_encoder = GRUSequenceEncoder(
            input_dim=phys_input_dim,
            hidden_dim=64,
            output_dim=phys_dim,
        )

        self.medication_encoder = GRUSequenceEncoder(
            input_dim=med_input_dim,
            hidden_dim=48,
            output_dim=med_dim,
        )

        self.temporal_fusion = nn.Sequential(
            nn.Linear(phys_dim + med_dim, fusion_dim),
            nn.ReLU(),
            nn.Dropout(0.15),
        )

        self.survival_head = nn.Sequential(
            nn.Linear(fusion_dim + valve_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(64, n_survival_bins),
        )

    def forward(
        self,
        x_static,
        x_phys,
        x_med,
        lengths,
    ):
        z_valve = self.valve_encoder(x_static)

        z_phys = self.physiology_encoder(
            x_phys,
            lengths,
        )

        z_med = self.medication_encoder(
            x_med,
            lengths,
        )

        z_patient = self.temporal_fusion(
            torch.cat(
                [z_phys, z_med],
                dim=1,
            )
        )

        fused = torch.cat(
            [z_patient, z_valve],
            dim=1,
        )

        hazard_logits = self.survival_head(fused)
        hazards = torch.sigmoid(hazard_logits)

        survival = torch.cumprod(
            1.0 - hazards + 1e-7,
            dim=1,
        )

        return {
            "hazard_logits": hazard_logits,
            "hazards": hazards,
            "survival": survival,
            "z_valve": z_valve,
            "z_phys": z_phys,
            "z_med": z_med,
            "z_patient": z_patient,
        }


## 8. Instantiate model and run a forward pass

In [ ]:
STATIC_DIM = batch["x_static"].shape[1]
PHYS_DIM = batch["x_phys"].shape[2]
MED_DIM = batch["x_med"].shape[2]

model = MultimodalValveSurvivalModel(
    static_input_dim=STATIC_DIM,
    phys_input_dim=PHYS_DIM,
    med_input_dim=MED_DIM,
).to(DEVICE)

print(model)

with torch.no_grad():
    outputs = model(
        batch["x_static"].to(DEVICE),
        batch["x_phys"].to(DEVICE),
        batch["x_med"].to(DEVICE),
        batch["lengths"].to(DEVICE),
    )

print("Hazards:", outputs["hazards"].shape)
print("Survival:", outputs["survival"].shape)
print("Valve embedding:", outputs["z_valve"].shape)
print("Patient embedding:", outputs["z_patient"].shape)

print("\nExample S(t):")
print(outputs["survival"][0].detach().cpu().numpy())


## 9. Discrete-time survival loss

For an event patient:

- intervals before the event must be survived,
- the event interval contributes `log(hazard)`.

For a censored patient:

- all fully observed intervals before censoring contribute survival terms,
- no event term is added.


In [ ]:
def duration_to_bin(duration_months):
    # Maps duration to 0...(N_SURVIVAL_BINS-1).
    idx = torch.floor(
        duration_months / 12.0
    ).long()

    return torch.clamp(
        idx,
        min=0,
        max=N_SURVIVAL_BINS - 1,
    )


def discrete_time_survival_loss(
    hazard_logits,
    duration,
    event,
):
    hazards = torch.sigmoid(hazard_logits)
    eps = 1e-7

    batch_size = hazards.shape[0]
    event_bin = duration_to_bin(duration)

    losses = []

    for i in range(batch_size):
        k = int(event_bin[i].item())

        # Survive all intervals strictly before k.
        if k > 0:
            survive_loss = -torch.log(
                1.0 - hazards[i, :k] + eps
            ).sum()
        else:
            survive_loss = torch.tensor(
                0.0,
                device=hazards.device,
            )

        if event[i] > 0.5:
            event_loss = -torch.log(
                hazards[i, k] + eps
            )

            loss_i = survive_loss + event_loss

        else:
            # Censored at duration:
            # require survival through interval k.
            censor_loss = -torch.log(
                1.0 - hazards[i, k] + eps
            )

            loss_i = survive_loss + censor_loss

        losses.append(loss_i)

    return torch.stack(losses).mean()


test_loss = discrete_time_survival_loss(
    outputs["hazard_logits"],
    batch["duration"].to(DEVICE),
    batch["event"].to(DEVICE),
)

print("Test loss:", float(test_loss))


## 10. Training loop

In [ ]:
def move_batch_to_device(batch, device):
    return {
        "x_static": batch["x_static"].to(device),
        "x_phys": batch["x_phys"].to(device),
        "x_med": batch["x_med"].to(device),
        "lengths": batch["lengths"].to(device),
        "duration": batch["duration"].to(device),
        "event": batch["event"].to(device),
    }


def run_epoch(
    model,
    loader,
    optimizer=None,
):
    training = optimizer is not None

    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_n = 0

    for batch in loader:
        b = move_batch_to_device(
            batch,
            DEVICE,
        )

        with torch.set_grad_enabled(training):
            out = model(
                b["x_static"],
                b["x_phys"],
                b["x_med"],
                b["lengths"],
            )

            loss = discrete_time_survival_loss(
                out["hazard_logits"],
                b["duration"],
                b["event"],
            )

            if training:
                optimizer.zero_grad()
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=5.0,
                )

                optimizer.step()

        n = b["x_static"].shape[0]

        total_loss += float(loss.detach()) * n
        total_n += n

    return total_loss / max(total_n, 1)


## 11. Train a first prototype

This is a **prototype training run**, not a final scientific experiment.

For a serious experiment we should later add:
- repeated seeds,
- hyperparameter tuning,
- calibration,
- C-index / time-dependent AUC,
- Brier score,
- bootstrap confidence intervals,
- ablations.


In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

N_EPOCHS = 30
PATIENCE = 6

best_state = None
best_val = float("inf")
patience_counter = 0

history = []

for epoch in range(1, N_EPOCHS + 1):
    train_loss = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
    )

    val_loss = run_epoch(
        model,
        val_loader,
        optimizer=None,
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train={train_loss:.4f} | "
        f"val={val_loss:.4f}"
    )

    if val_loss < best_val - 1e-4:
        best_val = val_loss
        best_state = copy.deepcopy(
            model.state_dict()
        )
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("Early stopping.")
        break

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
display(history_df)


## 12. Test loss + survival curves

In [ ]:
test_loss = run_epoch(
    model,
    test_loader,
    optimizer=None,
)

print("Test survival NLL:", round(test_loss, 4))


@torch.no_grad()
def predict_loader(model, loader):
    model.eval()

    rows = []

    for batch in loader:
        b = move_batch_to_device(
            batch,
            DEVICE,
        )

        out = model(
            b["x_static"],
            b["x_phys"],
            b["x_med"],
            b["lengths"],
        )

        survival = (
            out["survival"]
            .detach()
            .cpu()
            .numpy()
        )

        for i, pid in enumerate(batch["patient_id"]):
            row = {
                "Patient": pid,
                "duration_months": float(
                    batch["duration"][i]
                ),
                "event": int(
                    batch["event"][i]
                ),
            }

            for k in range(N_SURVIVAL_BINS):
                month = int(
                    SURVIVAL_BIN_EDGES[k + 1]
                )

                row[f"S_{month}m"] = float(
                    survival[i, k]
                )

            rows.append(row)

    return pd.DataFrame(rows)


test_predictions = predict_loader(
    model,
    test_loader,
)

display(test_predictions.head())


## 13. Simple structural sanity checks

Survival should be non-increasing over time for every patient.


In [ ]:
surv_cols = [
    c for c in test_predictions.columns
    if c.startswith("S_")
]

surv_array = test_predictions[surv_cols].to_numpy()

non_increasing = np.all(
    np.diff(surv_array, axis=1) <= 1e-6
)

print("All predicted survival curves non-increasing:", non_increasing)

assert non_increasing


## 14. Save model + preprocessing metadata

In [ ]:
OUTPUT_DIR = Path("model_outputs_v1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.save(
    model.state_dict(),
    OUTPUT_DIR / "multimodal_valve_survival_model_v1.pt",
)

test_predictions.to_csv(
    OUTPUT_DIR / "test_survival_predictions_v1.csv",
    index=False,
)

history_df.to_csv(
    OUTPUT_DIR / "training_history_v1.csv",
    index=False,
)

metadata = {
    "static_numeric": STATIC_NUMERIC,
    "static_categorical": STATIC_CATEGORICAL,
    "static_model_features": STATIC_FEATURES_MODEL,
    "physiology_features": PHYS_FEATURES,
    "medication_features": MED_FEATURES,
    "note_features_appended_to_physiology": NOTE_FEATURES,
    "time_scale": TIME_SCALE,
    "survival_bin_edges_months": SURVIVAL_BIN_EDGES.tolist(),
    "n_survival_bins": N_SURVIVAL_BINS,
    "architecture": {
        "valve_encoder": "MLP",
        "physiology_encoder": "GRU",
        "medication_encoder": "GRU",
        "temporal_fusion": "MLP",
        "survival_head": "discrete-time hazards",
    },
}

with open(
    OUTPUT_DIR / "model_metadata_v1.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Saved outputs to:", OUTPUT_DIR.resolve())


# What this notebook gives us

After this notebook runs successfully, we have a complete end-to-end prototype:

```text
Synthetic longitudinal multimodal dataset
        ↓
Patient-level split
        ↓
Leakage-safe preprocessing
        ↓
Static Valve MLP
Physiology GRU
Medication GRU
        ↓
Temporal Fusion
        ↓
Discrete-time Survival Head
        ↓
S(t)
```

## Next scientific steps

The next notebook should focus on **evaluation**, not on making the architecture more complicated.

Recommended Step 4:

- event / censor balance by split,
- C-index,
- Brier score,
- calibration,
- survival curves,
- ablation:
  - valve only,
  - physiology only,
  - medications only,
  - valve + physiology,
  - all modalities,
- explainability / feature importance,
- sensitivity to missingness.

Only after that should we consider:
- Transformer instead of GRU,
- raw clinical-note text encoder,
- attention-based multimodal fusion.
